# 10B · The Killer Applications — Retirement & Project NPV
### Financial Analytics — Module 10 · Lab 2

The engine from 10A, pointed at the two questions Monte Carlo answers better than any other tool on earth:

1. **"Will my money last?"** — the retirement corpus, sequence risk included
2. **"Is this project worth it?"** — MoneyMart's rollout NPV as a *distribution*, not a number

> 🛡️ **Bias check:** return assumptions below are stated, not estimated from a survivor-biased fund list. The retirement model ignores taxes/fees (stated simplification — both reduce corpora, so our answers are OPTIMISTIC bounds). Regime: a single (mu, sigma) for 35 years is itself a heroic regime assumption — carried openly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

---
## 1. The retirement question, honestly answered

**The plan:** Priya, 25, invests **₹20,000/month** (SIP) for 35 years, then retires at 60 and withdraws **₹1.5 lakh/month** (today's money; we work in real, inflation-adjusted terms throughout — simpler and more honest). Equity assumption: **7% real return, 17% volatility** (stated, arguable, adjustable).

**The deterministic lie first** — the number every calculator and every agent quotes:

In [ ]:
MU_R, SIGMA = 0.07, 0.17          # REAL annual return & vol
mu_m, sig_m = MU_R/12, SIGMA/np.sqrt(12)
SIP, WD = 20_000, 150_000
YRS_SAVE, YRS_SPEND = 35, 30       # save 25->60, spend 60->90

# The smooth-average world: same return every single month
corpus = 0.0
for m in range(YRS_SAVE*12):
    corpus = corpus*(1+mu_m) + SIP
det_corpus = corpus
for m in range(YRS_SPEND*12):
    corpus = corpus*(1+mu_m) - WD
print(f"Deterministic corpus at 60: Rs {det_corpus/1e7:.2f} crore")
print(f"Deterministic money at 90 : Rs {corpus/1e7:,.2f} crore  -> 'the plan works PERFECTLY'")

In [ ]:
# Now the truth: 10,000 lifetimes with RANDOM month-by-month returns
N = 10_000
months_total = (YRS_SAVE+YRS_SPEND)*12
R = mu_m + sig_m*rng.standard_normal((N, months_total))

corpus = np.zeros(N)
corpus_at_60 = np.zeros(N)
ruin_age = np.full(N, np.nan)

for m in range(months_total):
    if m < YRS_SAVE*12:
        corpus = corpus*(1+R[:,m]) + SIP
        if m == YRS_SAVE*12-1: corpus_at_60 = corpus.copy()
    else:
        corpus = corpus*(1+R[:,m]) - WD
        newly_ruined = (corpus <= 0) & np.isnan(ruin_age)
        ruin_age[newly_ruined] = 60 + (m - YRS_SAVE*12)/12
        corpus = np.maximum(corpus, 0)

print(f"Corpus at 60 - median Rs {np.median(corpus_at_60)/1e7:.2f} cr | 10th pct Rs {np.quantile(corpus_at_60,.1)/1e7:.2f} cr | 90th Rs {np.quantile(corpus_at_60,.9)/1e7:.2f} cr")
print(f"\nP(money lasts to 90) = {np.isnan(ruin_age).mean():.1%}")
print(f"Among failures, median ruin age: {np.nanmedian(ruin_age):.0f}")
print(f"\nSame assumptions as the 'perfect' plan. Randomness alone creates a {1-np.isnan(ruin_age).mean():.0%} failure rate.")

**Sit with that.** The deterministic calculator said the plan works *with room to spare*. Ten thousand honest lifetimes say it fails a material fraction of the time — same average return, same everything, just the acknowledgment that returns arrive in random order. This gap between the smooth-average story and the distribution of real outcomes is, arguably, the single most consequential fact in personal finance — and Monte Carlo is the only tool that shows it.

## 2. Why order matters: sequence-of-returns risk

In [ ]:
# Same 30 retirement years, same set of annual returns - just REVERSED order
good_early = np.array([0.12]*15 + [-0.02]*15)     # bull first, bear late
bad_early  = good_early[::-1]                      # bear first, bull late
print(f"Identical average return: {good_early.mean():.1%} both ways\n")

for name, seq in [("Good years FIRST", good_early), ("Bad years FIRST", bad_early)]:
    c = 3.0e7                                      # Rs 3 cr corpus, withdrawing 18L/yr
    for r in seq:
        c = c*(1+r) - 18e5
        c = max(c, 0)
    print(f"{name}: corpus after 30 yrs = Rs {c/1e7:5.2f} cr")

**Identical returns, opposite fates.** When you're *withdrawing*, early losses are lethal: each withdrawal in a down market sells a bigger slice of a shrinking corpus, and the recovery arrives to a portfolio too small to ride it. (During *accumulation* the effect flips — early crashes mean cheap SIP purchases.) This is **sequence risk**, it's why "average return" is nearly meaningless for retirees, and why real retirement planning is Monte Carlo planning.

### ✏️ Exercise 1
Find Priya's honest SIP: at what monthly investment does P(lasts to 90) reach 90%? (Loop over SIP values — the engine is already built.) Compare it to the deterministic calculator's answer and write the one-line sales-pitch critique.

---
## 3. Project NPV as a distribution — the tornado grows up

Module 4 drew a tornado (one assumption wiggling at a time). Module 7 warned that optimisers exploit point estimates. Here's the graduation: **wiggle everything at once, ten thousand times.** MoneyMart's 40-store rollout, three uncertain inputs:

In [ ]:
N = 10_000
INVEST = 120                                             # Rs cr, upfront
rev_growth = rng.normal(0.10, 0.04, N)                   # yearly revenue growth: uncertain
margin     = rng.normal(0.085, 0.015, N)                 # EBITDA-ish margin: uncertain
disc       = 0.115                                       # WACC held fixed (Exercise 2 frees it)

rev0 = 55.0                                              # Rs cr, year-1 revenue of the rollout
years = np.arange(1, 8)
# 7-year cash flows per path: revenue grows, margin applies; then discount and sum
revs = rev0 * np.cumprod(1 + np.tile(rev_growth, (7,1)).T, axis=1)
cfs  = revs * margin[:, None]
npv  = (cfs / (1+disc)**years).sum(axis=1) - INVEST

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(npv, bins=80, color="#EA580C", alpha=0.85)
ax.axvline(0, color="black", lw=1.2)
ax.axvline(np.median(npv), color="#2563EB", lw=1.5, ls="--", label=f"median Rs {np.median(npv):.0f} cr")
ax.set_title(f"NPV distribution: P(NPV < 0) = {(npv<0).mean():.0%} - the number the base case hides",
             loc="left", fontweight="bold")
ax.set_xlabel("NPV (Rs crore)"); ax.legend(); plt.tight_layout(); plt.show()

print(f"Base-case NPV (means plugged in): Rs {((rev0*np.cumprod([1.10]*7)*0.085)/(1+disc)**years).sum()-INVEST:.0f} cr")
print(f"Median simulated NPV:             Rs {np.median(npv):.0f} cr")
print(f"P(NPV < 0):                       {(npv<0).mean():.0%}")
print(f"5th percentile ('bad but plausible'): Rs {np.quantile(npv, .05):.0f} cr")

**The sentence for the board changes completely.** Before: *"NPV is ₹34 crore, approve."* After: *"Median NPV ₹34 crore, but a 1-in-5 chance of destroying value, and the plausible downside is −₹40 crore — do we have the balance sheet for that tail, and can we stage the rollout to buy the option to stop?"* That last clause is Module 7's optionality thinking, now with a probability attached — the two labs meeting.

### ✏️ Exercises
2. **Free the WACC:** make the discount rate uncertain too (`rng.normal(0.115, 0.01, N)`). How much does P(NPV<0) move? Compare its influence to margin's (correlate each input with NPV across paths — a *simulation tornado*).
3. **Correlated inputs, the subtle killer:** in recessions, growth AND margin fall together. Rebuild with correlation 0.6 between them (hint: `g = base; m = 0.6*standardised_g + sqrt(1-0.36)*independent`). Does P(NPV<0) rise or fall — and why does independence flatter almost every business case?

---
*AI disclosure: ______*

In [ ]:
# workspace
